# Analysis, first version

An earlier version of the analysis notebook, replaced by 08_analysis. Merges the outputs of the training notebooks and runs analysis.py and gradcam.py.

In [ ]:
%%writefile /kaggle/working/run_experiments.py
"""
Training and evaluation for every configuration in the paper: the five
architectures, the A0-A4 ablation on ResNet50 and the oversampling runs.
All of them use the same budget (at most 10 epochs, patience 3).

Each run writes results/<run_name>.json and logits/<run_name>.npz. A run whose
JSON already exists is skipped, so an interrupted session can be restarted.

Examples:
    !python run_experiments.py --arch resnet50 --variant A0 --seeds 0 1 2 3 4
    !python run_experiments.py --arch resnet50 --variant A4 --seeds 0 1 2 3 4
    !python run_experiments.py --arch efficientnet_b0 --variant A0 --seeds 0 1 2 3 4
    !python run_experiments.py --arch resnet50 --variant A0 --seeds 0 --smoke
"""
import argparse, json, os, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

# ---------------------------------------------------------------- paths
# Defaults are the Kaggle input paths; set the environment variables to run elsewhere.
PLANTVILLAGE_ROOT = Path(os.environ.get("PV_ROOT", "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color"))
SPLIT_ROOT = Path(os.environ.get("SPLIT_ROOT", "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/splits-20260929T040001Z-1-001/splits"))
TRAIN_VAL_FILE = Path(os.environ.get("TRAIN_VAL_FILE", "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/train_validation_split.npz"))
OUT = Path(os.environ.get("OUT_DIR", "/kaggle/working/revision"))

# ---------------------------------------------------------------- fixed protocol (Table 1 in the paper)
IMAGE_SIZE, BATCH_SIZE, NUM_CLASSES = 224, 32, 38
LR, WEIGHT_DECAY, MAX_EPOCHS, PATIENCE = 1e-3, 1e-4, 10, 3
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
TEST_NOISE_SEED = 12345  # fixed, so every model sees the same noisy test images

TEST_CONDITIONS = [  # (family, severity, parameter)
    ("brightness", "mild", 0.80), ("brightness", "moderate", 0.60), ("brightness", "severe", 0.40),
    ("gaussian_noise", "mild", 0.03), ("gaussian_noise", "moderate", 0.07), ("gaussian_noise", "severe", 0.12),
    ("gaussian_blur", "mild", 1.0), ("gaussian_blur", "moderate", 2.0), ("gaussian_blur", "severe", 3.0),
]
VARIANTS = {  # training corruption families per ablation variant
    "A0": [], "A1": ["blur"], "A2": ["noise"], "A3": ["brightness"], "A4": ["blur", "noise", "brightness"],
}


# ---------------------------------------------------------------- reproducibility
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class RecordingSampler(WeightedRandomSampler):
    """Keeps the indices drawn in the last epoch so the distinct images can be counted."""
    def __iter__(self):
        self.last = list(super().__iter__())
        return iter(self.last)


def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)


# ---------------------------------------------------------------- data: official PlantVillage split, group-aware validation indices
def load_splits():
    read = lambda f: [l.strip() for l in open(f, encoding="utf-8") if l.strip()]
    train_all, test = read(SPLIT_ROOT / "color_train.txt"), read(SPLIT_ROOT / "color_test.txt")
    idx = np.load(TRAIN_VAL_FILE)
    lookup = {p.name: str(p) for d in PLANTVILLAGE_ROOT.iterdir() if d.is_dir() for p in d.iterdir()}

    def resolve(rel):
        name = Path(rel).name
        return lookup.get(name) or lookup[re.sub(r"\s+(\.[^.]+)$", r"\1", name)]

    rows = [(resolve(r), Path(r).parent.name, s)
            for s, paths in [("train", [train_all[i] for i in idx["train_idx"]]),
                             ("val", [train_all[i] for i in idx["val_idx"]]),
                             ("test", test)]
            for r in paths]
    df = pd.DataFrame(rows, columns=["path", "class_name", "split"])
    classes = sorted(df.class_name.unique())
    df["label"] = df.class_name.map({c: i for i, c in enumerate(classes)})
    sizes = df.split.value_counts()
    assert len(classes) == 38 and (sizes["train"], sizes["val"], sizes["test"]) == (39_207, 4_389, 10_709)
    return {s: df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"]}


class ImageDataset(Dataset):
    def __init__(self, df, transform, pre=None):
        self.df, self.transform, self.pre = df, transform, pre  # pre: corruption on the 256x256 original

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        img = Image.open(self.df.path[i]).convert("RGB")
        if self.pre is not None:
            img = self.pre(img, i)
        return self.transform(img), int(self.df.label[i])


# ---------------------------------------------------------------- corruptions
def add_noise(img, std, rng):
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = np.clip(arr + rng.normal(0.0, std, arr.shape), 0.0, 1.0)
    return Image.fromarray((arr * 255.0).astype(np.uint8))


def test_corruption(family, value):
    if family == "brightness":
        return lambda img, i: ImageEnhance.Brightness(img).enhance(value)
    if family == "gaussian_blur":
        return lambda img, i: img.filter(ImageFilter.GaussianBlur(radius=value))
    # noise seeded by image index: identical corrupted test set for every model and seed (paired tests)
    return lambda img, i: add_noise(img, value, np.random.default_rng(TEST_NOISE_SEED + i))


def train_corruption(families):
    def apply(img):  # each family is applied with p = 0.5
        if "blur" in families and random.random() < 0.5:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.4, 1.2)))
        if "noise" in families and random.random() < 0.5:
            img = add_noise(img, random.uniform(0.02, 0.06), np.random)
        if "brightness" in families and random.random() < 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.65, 1.35))
        return img
    return apply


def make_transforms(families, corrupt_before_resize):
    base = [transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.05), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    resize = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
    corrupt = [transforms.Lambda(train_corruption(families))] if families else []
    order = corrupt + [resize] if corrupt_before_resize else [resize] + corrupt
    evaluate = transforms.Compose([resize, transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return transforms.Compose(order + base), evaluate


# ---------------------------------------------------------------- models: frozen backbone, trainable head
def build_model(arch):
    head = lambda n: nn.Sequential(nn.Dropout(0.2), nn.Linear(n, NUM_CLASSES))
    if arch == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = head(m.fc.in_features); trainable = m.fc
    elif arch == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = head(m.classifier.in_features); trainable = m.classifier
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier = head(m.classifier[1].in_features); trainable = m.classifier
    elif arch == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        m.classifier = head(m.classifier[1].in_features); trainable = m.classifier
    elif arch == "mobilenet_v3_large":
        # the whole classifier (960 -> 1280 -> 38) is trained, not only the last layer
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES); trainable = m.classifier
    else:
        raise ValueError(arch)
    for p in m.parameters():
        p.requires_grad = False
    for p in trainable.parameters():
        p.requires_grad = True
    return m, trainable


# ---------------------------------------------------------------- train / evaluate
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    logits, labels = [], []
    for x, y in loader:
        logits.append(model(x.to(device, non_blocking=True)).float().cpu()); labels.append(y)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy()


def scores(logits, labels):
    pred = logits.argmax(1)
    return {"acc": 100 * accuracy_score(labels, pred), "macro_f1": 100 * f1_score(labels, pred, average="macro", zero_division=0)}


def run(arch, variant, seed, oversample, corrupt_before_resize, data, device, smoke):
    name = f"{arch}_{variant}{'_os' if oversample else ''}_seed{seed}"
    res_path, logit_path = OUT / "results" / f"{name}.json", OUT / "logits" / f"{name}.npz"
    if res_path.exists():
        print(f"skip {name} (done)"); return
    set_seed(seed)
    train_df, val_df, test_df = data["train"], data["val"], data["test"]
    if smoke:  # quick pipeline check only, numbers are meaningless
        train_df, val_df, test_df = (d.sample(500, random_state=0).reset_index(drop=True) for d in (train_df, val_df, test_df))

    train_tf, eval_tf = make_transforms(VARIANTS[variant], corrupt_before_resize)
    g = torch.Generator().manual_seed(seed)
    loader_kw = dict(batch_size=BATCH_SIZE, num_workers=min(4, os.cpu_count() or 2), pin_memory=True, worker_init_fn=seed_worker)
    if oversample:  # weight = 1 / sqrt(class proportion)
        prop = train_df.label.map(train_df.label.value_counts(normalize=True)).values
        w = 1.0 / np.sqrt(prop); w = w / w.mean()
        sampler = RecordingSampler(torch.tensor(w, dtype=torch.double), len(train_df), replacement=True, generator=g)
        train_loader = DataLoader(ImageDataset(train_df, train_tf), sampler=sampler, generator=g, **loader_kw)
    else:
        train_loader = DataLoader(ImageDataset(train_df, train_tf), shuffle=True, generator=g, **loader_kw)
    val_loader = DataLoader(ImageDataset(val_df, eval_tf), shuffle=False, **loader_kw)

    model, head = build_model(arch)
    model.to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    torch.cuda.reset_peak_memory_stats(device)
    t0, best_f1, best_epoch, best_state, wait, history = time.time(), -1.0, 0, None, 0, []
    # progress is saved after every epoch, so a dropped session loses at most one epoch
    resume_path, start, prev_seconds, stopped = OUT / "resume" / f"{name}.pt", 1, 0.0, False
    if resume_path.exists() and not smoke:
        r = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.load_state_dict(r["model"]); opt.load_state_dict(r["opt"]); g.set_state(r["g"])
        torch.set_rng_state(r["torch_rng"]); np.random.set_state(r["np_rng"]); random.setstate(r["py_rng"])
        best_f1, best_epoch, best_state, wait = r["best_f1"], r["best_epoch"], r["best_state"], r["wait"]
        history, prev_seconds, stopped, start = r["history"], r["seconds"], r["stopped"], r["epoch"] + 1
        print(f"resume {name} after epoch {r['epoch']}")
    for epoch in range(start, (1 if smoke else MAX_EPOCHS) + 1):
        if stopped:
            break
        # train mode: only the head gets gradients, but BatchNorm running statistics
        # in the backbone still update
        model.train()
        for x, y in train_loader:
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(x.to(device, non_blocking=True)), y.to(device, non_blocking=True))
            loss.backward(); opt.step()
        val = scores(*predict(model, val_loader, device))
        unique = len(set(sampler.last)) if oversample else len(train_df)
        history.append({"epoch": epoch, "unique_train_images": unique, **val})
        print(f"{name} epoch {epoch}: val acc {val['acc']:.2f} macro F1 {val['macro_f1']:.2f}")
        if val["macro_f1"] > best_f1:
            best_f1, best_epoch, wait = val["macro_f1"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        stopped = wait >= PATIENCE
        if not smoke:
            resume_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"epoch": epoch, "model": model.state_dict(), "opt": opt.state_dict(), "g": g.get_state(),
                        "torch_rng": torch.get_rng_state(), "np_rng": np.random.get_state(), "py_rng": random.getstate(),
                        "best_f1": best_f1, "best_epoch": best_epoch, "best_state": best_state, "wait": wait,
                        "history": history, "seconds": prev_seconds + time.time() - t0, "stopped": stopped},
                       str(resume_path) + ".tmp")
            os.replace(str(resume_path) + ".tmp", resume_path)  # atomic, so a disconnect mid-save can't corrupt it
        if stopped:
            break
    train_seconds = prev_seconds + time.time() - t0
    model.load_state_dict(best_state)

    results, all_logits = {}, {}
    conditions = [("clean", "none", None)] + TEST_CONDITIONS
    for family, severity, value in conditions:
        pre = None if family == "clean" else test_corruption(family, value)
        loader = DataLoader(ImageDataset(test_df, eval_tf, pre), shuffle=False, **loader_kw)
        logits, labels = predict(model, loader, device)
        key = family if family == "clean" else f"{family}_{severity}"
        results[key] = scores(logits, labels); all_logits[key] = logits.astype(np.float16)
    corr = [results[f"{f}_{s}"] for f, s, _ in TEST_CONDITIONS]

    out = {
        "run": name, "arch": arch, "variant": variant, "seed": seed, "oversample": oversample,
        "corrupt_before_resize": corrupt_before_resize, "smoke": smoke,
        "total_params": total, "trainable_params": trainable,
        "epochs_trained": len(history), "best_epoch": best_epoch, "history": history,
        "train_seconds": round(train_seconds, 1),
        "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated(device) / 2**20, 1),
        "results": results,
        "mean_corr_acc": float(np.mean([r["acc"] for r in corr])),
        "mean_corr_macro_f1": float(np.mean([r["macro_f1"] for r in corr])),
        "worst_acc": float(min(r["acc"] for r in corr)),
        "worst_macro_f1": float(min(r["macro_f1"] for r in corr)),
        "mean_unique_train_images": float(np.mean([h["unique_train_images"] for h in history])),
        "torch": torch.__version__,
    }
    for d in ("results", "logits", "checkpoints"):
        (OUT / d).mkdir(parents=True, exist_ok=True)
    np.savez_compressed(logit_path, labels=labels, **all_logits)
    torch.save(best_state, OUT / "checkpoints" / f"{name}.pth")
    json.dump(out, open(res_path, "w"), indent=2)  # written last: its presence means the run is complete
    if resume_path.exists():
        resume_path.unlink()
    print(f"done {name}: clean {results['clean']['acc']:.2f} | mean corr {out['mean_corr_acc']:.2f} | "
          f"worst {out['worst_acc']:.2f} | {train_seconds/60:.1f} min | {out['peak_gpu_mem_mb']} MB")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", required=True,
                    choices=["resnet50", "densenet121", "efficientnet_b0", "efficientnet_v2_s", "mobilenet_v3_large"])
    ap.add_argument("--variant", default="A0", choices=list(VARIANTS))
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2, 3, 4])
    ap.add_argument("--oversample", action="store_true")
    ap.add_argument("--corrupt_before_resize", action="store_true",
                    help="apply training corruptions before resizing, like the test corruptions")
    ap.add_argument("--smoke", action="store_true", help="500 images, 1 epoch, to check the pipeline")
    a = ap.parse_args()
    if a.smoke:
        OUT = OUT / "smoke"
    device = torch.device("cuda")
    data = load_splits()
    for s in a.seeds:
        run(a.arch, a.variant, s, a.oversample, a.corrupt_before_resize, data, device, a.smoke)

In [ ]:
%%writefile /kaggle/working/analysis.py
"""
Summary tables, paired tests and calibration from the outputs of run_experiments.py.
No training: it reads results/*.json and logits/*.npz under OUT_DIR.

    python analysis.py            # writes OUT_DIR/analysis/

Outputs
  summary.csv / summary_table.tex        mean ± SD over seeds per configuration
  per_condition.csv / per_condition.tex  accuracy of every configuration under every condition
  paired_tests.csv                       paired t-tests against a reference, Holm-corrected
  calibration.csv / calibration.tex      ECE, MCE and Brier score, clean and corrupted
  reliability_*.png                      reliability diagrams, 300 dpi
  compute.csv                            training time and peak GPU memory
"""
import json, os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path(os.environ.get("OUT_DIR", "/kaggle/working/revision"))
OUT = ROOT / "analysis"
N_BINS = 10  # equal-width confidence bins
CONDITIONS = ["clean"] + [f"{f}_{s}" for f in ("brightness", "gaussian_noise", "gaussian_blur")
                          for s in ("mild", "moderate", "severe")]
METRICS = ["clean_acc", "clean_macro_f1", "mean_corr_acc", "mean_corr_macro_f1", "worst_acc", "worst_macro_f1"]


def config_name(r):
    return f"{r['arch']}_{r['variant']}" + ("_os" if r["oversample"] else "")


def load_runs():
    rows = []
    for f in sorted((ROOT / "results").glob("*.json")):
        r = json.load(open(f))
        if r.get("smoke"):
            continue
        row = {"config": config_name(r), "run": r["run"], "seed": r["seed"],
               "clean_acc": r["results"]["clean"]["acc"], "clean_macro_f1": r["results"]["clean"]["macro_f1"],
               **{k: r[k] for k in METRICS[2:]},
               "train_min": r["train_seconds"] / 60, "peak_gpu_mb": r["peak_gpu_mem_mb"],
               "epochs": r["epochs_trained"], "best_epoch": r["best_epoch"]}
        row.update({f"acc_{c}": r["results"][c]["acc"] for c in CONDITIONS})
        rows.append(row)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- calibration
def calibration(logits, labels):
    z = logits.astype(np.float64)
    p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == labels).astype(float)
    edges = np.linspace(0, 1, N_BINS + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=False), 0, N_BINS - 1)  # last bin includes 1.0
    ece, mce, bins = 0.0, 0.0, []
    for b in range(N_BINS):
        m = idx == b
        if m.any():
            gap = abs(correct[m].mean() - conf[m].mean())
            ece += m.mean() * gap; mce = max(mce, gap)
            bins.append((conf[m].mean(), correct[m].mean(), int(m.sum())))
    onehot = np.eye(p.shape[1])[labels]
    brier = ((p - onehot) ** 2).sum(1).mean()
    return {"ece": 100 * ece, "mce": 100 * mce, "brier": brier, "mean_conf": 100 * conf.mean(),
            "acc": 100 * correct.mean()}, bins


def calibration_table(runs):
    rows = []
    for _, r in runs.iterrows():
        z = np.load(ROOT / "logits" / f"{r['run']}.npz")
        for c in CONDITIONS:
            m, _ = calibration(z[c], z["labels"])
            rows.append({"config": r["config"], "seed": r["seed"], "condition": c, **m})
    return pd.DataFrame(rows)


def reliability_plot(run_a, run_b, condition, path):
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.4), sharey=True)
    for ax, run in zip(axes, (run_a, run_b)):
        z = np.load(ROOT / "logits" / f"{run}.npz")
        m, bins = calibration(z[condition], z["labels"])
        conf, acc, n = zip(*bins)
        ax.plot([0, 1], [0, 1], "k--", lw=0.8)
        ax.bar(conf, acc, width=0.08, edgecolor="black", alpha=0.7)
        ax.set_title(f"{run.split('_seed')[0]}\nECE {m['ece']:.2f}%, MCE {m['mce']:.2f}%", fontsize=9)
        ax.set_xlabel("Mean confidence", fontsize=9); ax.tick_params(labelsize=8)
    axes[0].set_ylabel("Accuracy", fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


# ---------------------------------------------------------------- statistics
def holm(pvals):
    order = np.argsort(pvals); adj = np.empty(len(pvals)); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(pvals) - rank) * pvals[i]); adj[i] = min(1.0, running)
    return adj


def paired_tests(runs, pairs, metrics):
    rows = []
    for ref, other in pairs:
        a = runs[runs.config == ref].set_index("seed"); b = runs[runs.config == other].set_index("seed")
        seeds = a.index.intersection(b.index)
        if len(seeds) < 3:
            continue
        for m in metrics:
            d = b.loc[seeds, m] - a.loc[seeds, m]
            t, p = stats.ttest_rel(b.loc[seeds, m], a.loc[seeds, m])
            half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
            rows.append({"reference": ref, "compared": other, "metric": m, "n_seeds": len(seeds),
                         "mean_diff_pp": d.mean(), "ci95_low": d.mean() - half, "ci95_high": d.mean() + half,
                         "t": t, "p": p})
    df = pd.DataFrame(rows)
    if len(df):
        df["p_holm"] = holm(np.nan_to_num(df.p.values, nan=1.0))  # nan only if all seeds tie
        df["sig"] = np.where(df.p_holm < 0.05, "*", "n.s.")
    return df


# ---------------------------------------------------------------- tables
def fmt(m, s, nd):
    # SD is undefined with one seed: print the mean alone instead of NaN
    return m.apply(lambda c: c.map(lambda v: f"{v:.{nd}f}")).where(s.isna(),
           m.apply(lambda c: c.map(lambda v: f"{v:.{nd}f}")) + " ± " + s.apply(lambda c: c.map(lambda v: f"{v:.{nd}f}")))


def mean_sd(df, cols):
    g = df.groupby("config")[cols]
    return fmt(g.mean(), g.std(ddof=1), 2), g.size()


def booktabs(df, caption, path):
    cols = "l" + "c" * len(df.columns)
    body = df.to_latex(escape=True, column_format=cols)  # pandas emits \toprule/\midrule/\bottomrule (three-line table)
    path.write_text(f"% {caption}\n% needs \\usepackage{{booktabs}}\n" + body)


if __name__ == "__main__":
    OUT.mkdir(parents=True, exist_ok=True)
    runs = load_runs()
    assert len(runs), "no finished runs found"
    print(runs.groupby("config").size().rename("seeds").to_string())

    summary, n = mean_sd(runs, METRICS)
    summary.insert(0, "seeds", n)
    summary.to_csv(OUT / "summary.csv"); booktabs(summary, "Mean ± SD over seeds", OUT / "summary_table.tex")

    per_cond, _ = mean_sd(runs, [f"acc_{c}" for c in CONDITIONS])
    per_cond.columns = CONDITIONS
    per_cond.to_csv(OUT / "per_condition.csv"); booktabs(per_cond, "Accuracy per condition", OUT / "per_condition.tex")

    runs.groupby("config")[["train_min", "peak_gpu_mb", "epochs", "best_epoch"]].agg(["mean", "std"]).round(2) \
        .to_csv(OUT / "compute.csv")

    # Ablation vs A0, oversampling vs its base, architectures vs ResNet50
    pairs = [("resnet50_A0", f"resnet50_A{k}") for k in range(1, 5)]
    pairs += [("resnet50_A0", "resnet50_A0_os"), ("resnet50_A4", "resnet50_A4_os")]
    pairs += [(f"{a}_A0", f"{a}_A0_os") for a in ("densenet121", "efficientnet_b0", "efficientnet_v2_s")]
    pairs += [("resnet50_A0", f"{a}_A0") for a in ("densenet121", "efficientnet_b0", "efficientnet_v2_s", "mobilenet_v3_large")]
    tests = paired_tests(runs, pairs, METRICS)
    tests.round(4).to_csv(OUT / "paired_tests.csv", index=False)
    if len(tests):
        print(tests[["reference", "compared", "metric", "mean_diff_pp", "p_holm", "sig"]].round(3).to_string(index=False))

    cal = calibration_table(runs)
    cal.to_csv(OUT / "calibration_all.csv", index=False)
    g = cal.groupby(["config", "condition"])[["ece", "mce", "brier"]]
    cal_summary = fmt(g.mean(), g.std(ddof=1), 3)
    cal_summary.to_csv(OUT / "calibration.csv")
    booktabs(cal_summary.loc[cal_summary.index.get_level_values(1).isin(
        ["clean", "gaussian_noise_severe", "gaussian_blur_severe", "brightness_severe"])],
        "Calibration, clean and severe corruptions", OUT / "calibration.tex")

    for cond in ("clean", "gaussian_blur_severe", "gaussian_noise_severe"):
        if {"resnet50_A0_seed0", "resnet50_A4_seed0"} <= set(runs.run):
            reliability_plot("resnet50_A0_seed0", "resnet50_A4_seed0", cond, OUT / f"reliability_A0_vs_A4_{cond}.png")
    print(f"written to {OUT}")

In [ ]:
%%writefile /kaggle/working/gradcam.py
"""
Grad-CAM on a fixed random sample of test images, using a checkpoint saved by
run_experiments.py.

    !python gradcam.py --run resnet50_A0_seed0 --n_correct 50 --n_wrong 50
    !python gradcam.py --run resnet50_A4_seed0 --n_correct 50 --n_wrong 50 --segmented_root /kaggle/input/.../segmented

Outputs in revision/gradcam/<run>/: grid figures at 300 dpi and samples.csv.
With --segmented_root (PlantVillage "segmented" version, black background) it
also reports the share of Grad-CAM energy that falls on the leaf, which turns
the qualitative maps into a number over the whole sample.
"""
import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

from run_experiments import OUT, build_model, load_splits, make_transforms


class GradCAM:
    def __init__(self, model, layer):
        self.model, self.act, self.grad = model, None, None
        self.h1 = layer.register_forward_hook(lambda m, i, o: setattr(self, "act", o.detach()))
        self.h2 = layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grad", go[0].detach()))

    def __call__(self, x, cls):
        self.model.zero_grad()
        out = self.model(x)
        out[:, cls].sum().backward()
        cam = F.relu((self.grad.mean((2, 3), keepdim=True) * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False).squeeze()
        return ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy()


def target_layer(model, arch):
    if arch == "resnet50":
        return model.layer4
    if arch == "densenet121":
        return model.features
    return model.features[-1]  # EfficientNet-B0, EfficientNetV2-S, MobileNetV3


def leaf_mask(path, seg_lookup):
    # segmented PlantVillage files carry a "_final_masked" suffix
    p = Path(path)
    seg = seg_lookup.get(f"{p.stem}_final_masked{p.suffix}") or seg_lookup.get(p.name)
    if seg is None:
        return None
    m = np.asarray(Image.open(seg).convert("L").resize((224, 224))) > 10
    return m


def grid(items, title, path):
    rows = len(items)
    fig, axes = plt.subplots(rows, 2, figsize=(4.2, 2.1 * rows))
    axes = np.atleast_2d(axes)
    for ax_row, it in zip(axes, items):
        ax_row[0].imshow(it["img"]); ax_row[1].imshow(it["img"]); ax_row[1].imshow(it["cam"], cmap="jet", alpha=0.45)
        ax_row[0].set_title(f"True: {it['true']}", fontsize=8)
        ax_row[1].set_title(f"Pred: {it['pred']} ({it['conf']:.1f}%)", fontsize=8)
        for a in ax_row:
            a.axis("off")
    fig.suptitle(title, fontsize=8); fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, help="e.g. resnet50_A0_seed0")
    ap.add_argument("--n_correct", type=int, default=50)
    ap.add_argument("--n_wrong", type=int, default=50)
    ap.add_argument("--segmented_root", default=None)
    a = ap.parse_args()

    arch = next(x for x in ["efficientnet_v2_s", "efficientnet_b0", "mobilenet_v3_large", "densenet121", "resnet50"]
                if a.run.startswith(x))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 100 images run fine on a CPU
    model, _ = build_model(arch)
    model.load_state_dict(torch.load(OUT / "checkpoints" / f"{a.run}.pth", map_location="cpu"))
    model.to(device).eval()
    cam_fn = GradCAM(model, target_layer(model, arch))

    test = load_splits()["test"]
    _, eval_tf = make_transforms([], False)
    z = np.load(OUT / "logits" / f"{a.run}.npz")
    logits, labels = z["clean"].astype(np.float32), z["labels"]
    assert len(labels) == len(test) and (labels == test.label.values).all(), "logits do not match test order"
    probs = torch.softmax(torch.tensor(logits), 1).numpy()
    pred = probs.argmax(1)

    rng = np.random.default_rng(0)  # fixed sample, listed in samples.csv
    correct_idx = rng.choice(np.where(pred == labels)[0], a.n_correct, replace=False)
    wrong_pool = np.where(pred != labels)[0]
    wrong_idx = rng.choice(wrong_pool, min(a.n_wrong, len(wrong_pool)), replace=False)
    classes = sorted(test.class_name.unique())

    seg_lookup = {}
    if a.segmented_root:
        seg_lookup = {p.name: str(p) for d in Path(a.segmented_root).iterdir() if d.is_dir() for p in d.iterdir()}

    out_dir = OUT / "gradcam" / a.run
    out_dir.mkdir(parents=True, exist_ok=True)
    records, shown = [], {"correct": [], "wrong": []}
    for kind, idx in (("correct", correct_idx), ("wrong", wrong_idx)):
        for i in idx:
            img = Image.open(test.path[i]).convert("RGB")
            cam = cam_fn(eval_tf(img).unsqueeze(0).to(device), int(pred[i]))
            rec = {"index": int(i), "kind": kind, "true": classes[labels[i]], "pred": classes[pred[i]],
                   "conf": 100 * probs[i].max()}
            mask = leaf_mask(test.path[i], seg_lookup) if seg_lookup else None
            if mask is not None:
                rec["cam_on_leaf"] = float((cam * mask).sum() / (cam.sum() + 1e-8))
            records.append(rec)
            if len(shown[kind]) < 60:
                shown[kind].append({**rec, "img": np.asarray(img.resize((224, 224))), "cam": cam})
    df = pd.DataFrame(records)
    df.to_csv(out_dir / "samples.csv", index=False)

    for kind, items in shown.items():
        for page in range(0, len(items), 10):  # 10 images per figure keeps text readable at 8 pt
            grid(items[page:page + 10], f"{a.run}: {kind} predictions",
                 out_dir / f"{kind}_{page // 10 + 1}.png")
    if "cam_on_leaf" in df:
        print(df.groupby("kind")["cam_on_leaf"].describe().round(3).to_string())
    print(f"written to {out_dir}")

In [ ]:
%cd /kaggle/working

In [ ]:
# add the outputs of the training notebooks as inputs first (Add Input > Your Work)
!mkdir -p /kaggle/working/revision
!for d in $(find /kaggle/input -maxdepth 3 -type d -name revision); do cp -rn $d/* /kaggle/working/revision/; done
!ls /kaggle/working/revision/results | wc -l

In [ ]:
!python analysis.py

In [ ]:
!python gradcam.py --run resnet50_A0_seed0 --n_correct 50 --n_wrong 50
!python gradcam.py --run resnet50_A4_seed0 --n_correct 50 --n_wrong 50

In [ ]:
!cd /kaggle/working && zip -qr revision_analysis.zip revision/analysis revision/gradcam && ls -lh revision_analysis.zip